# LP3 Practical 10 – Bank Customer Churn Prediction using ANN
**Aim:** Build a neural-network classifier to predict whether a bank customer will leave (Exited = 1) or stay (Exited = 0).

**Dataset:** `Churn_Modelling - Copy.csv`

**Requirement:** TensorFlow/Keras. If TensorFlow is not installed, run `%pip install tensorflow` in a separate notebook cell, then restart the kernel.

## Cell 1 – Import libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report


## Cell 2 – Load and inspect dataset

In [ ]:
from pathlib import Path

def get_dataset_path(filename):
    candidates = [
        Path.cwd() / "Datasets" / filename,
        Path.cwd().parent / "Datasets" / filename
    ]
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError(
        f"Could not find {filename}. Keep the Datasets folder beside the Notebooks folder."
    )

DATA_PATH = get_dataset_path("Churn_Modelling - Copy.csv")
df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
display(df.head())
print(df["Exited"].value_counts())


## Cell 3 – Separate features and target

In [ ]:
# Remove identifier/name columns because they do not provide useful model input.
X = df.drop(columns=["RowNumber", "CustomerId", "Surname", "Exited"])
y = df["Exited"]

numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)


## Cell 4 – Split data and preprocess

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# Keras dense layers expect regular numeric arrays.
X_train_processed = X_train_processed.astype("float32")
X_test_processed = X_test_processed.astype("float32")

print("Training data shape:", X_train_processed.shape)
print("Testing data shape:", X_test_processed.shape)


## Cell 5 – Import TensorFlow and build ANN

In [ ]:
# If this import fails, run this in a separate cell:
# %pip install tensorflow
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.callbacks import EarlyStopping

tf.random.set_seed(42)

model = Sequential([
    Input(shape=(X_train_processed.shape[1],)),
    Dense(16, activation="relu"),
    Dense(8, activation="relu"),
    Dense(1, activation="sigmoid")
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()


## Cell 6 – Train the ANN

In [ ]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=4,
    restore_best_weights=True
)

history = model.fit(
    X_train_processed,
    y_train,
    epochs=30,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)


## Cell 7 – Plot training accuracy and loss

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(history.history["accuracy"], label="Training Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("ANN Accuracy")
plt.legend()
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("ANN Loss")
plt.legend()
plt.show()


## Cell 8 – Predict and evaluate

In [ ]:
probabilities = model.predict(X_test_processed)
predictions = (probabilities >= 0.5).astype(int).ravel()

print("Accuracy:", accuracy_score(y_test, predictions))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, predictions))
print("\nClassification Report:")
print(classification_report(y_test, predictions, zero_division=0))


## Sample output / observation
- `Exited = 1`: customer is predicted to leave.
- `Exited = 0`: customer is predicted to stay.
- The notebook prints accuracy, confusion matrix and classification report.
- Neural-network results can vary slightly with training environment and library version.